# Витрина транзакций ТЭ — 7. Два первых пункта анализа

AI-14135. Пункты 1 и 2: сравнение распределений и косинус портретов. Isolation Forest и кластеризация здесь не считаются.

Вход — уже собранный профиль `sandbox_ai.tmp_shestopalov_te_profile`, одна строка на ИНН. Операции заново не читаются. p10 и p90 чека в профиле нет, тесты идут по остальным полям портрета.

Группа меньше 30 клиентов в сравнение распределений и в косинус не входит. Клиенты без кода точки (`mcc_code = не заполнен`) убираются до расчётов: это пустой код, не вид деятельности.

Расчёт лежит в самой тетрадке. Отдельный файл рядом не нужен.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import jensenshannon

MIN_CLIENTS = 30
COMFORT_CLIENTS = 100
RANDOM_STATE = 23

METRICS = [
    "tx_count",
    "turnover",
    "avg_amount",
    "median_amount",
    "std_amount",
    "p10_amount",
    "p90_amount",
    "avg_interval",
    "median_interval",
    "success_rate",
    "active_days",
    "unique_bins",
    "unique_card_types",
]

# Эти две метрики задают прокси размера. Их убирают из тестов размера только при business_size_is_proxy = 1.
SIZE_BUILT_FROM = {"tx_count", "turnover"}

SLICES = [
    ("MCC", "mcc_code"),
    ("MCC + регион", "slice_mcc_region"),
    ("MCC + размер бизнеса", "slice_mcc_size"),
    ("Группа MCC", "mcc_group"),
    ("Основной ОКВЭД", "okved"),
    ("Основной ОКВЭД, класс", "okved_class"),
]

ASSOCIATIONS = [
    ("Группа MCC x регион", "mcc_group", "region_group"),
    ("Группа MCC x размер бизнеса", "mcc_group", "business_size"),
    ("ОКВЭД класс x регион", "okved_class", "region_group"),
    ("ОКВЭД класс x размер бизнеса", "okved_class", "business_size"),
]

def enough_label(n: int) -> str:
    if n >= COMFORT_CLIENTS:
        return "достаточно"
    if n >= MIN_CLIENTS:
        return "на грани"
    return "мало"


def coverage_table(profile: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for slice_name, col in SLICES:
        if col not in profile.columns:
            continue
        counts = profile[col].fillna("не заполнен").astype(str).value_counts()
        total = int(counts.sum())
        for value, n in counts.items():
            rows.append(
                {
                    "slice_name": slice_name,
                    "slice_value": value,
                    "client_cnt": int(n),
                    "client_share": int(n) / total if total else np.nan,
                    "enough": enough_label(int(n)),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["slice_name", "slice_value", "client_cnt", "client_share", "enough"])
    return pd.DataFrame(rows).sort_values(["slice_name", "client_cnt"], ascending=[True, False])


def bh_adjust(pvalues: pd.Series) -> pd.Series:
    p = pvalues.to_numpy(dtype=float)
    out = np.full(len(p), np.nan)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return pd.Series(out, index=pvalues.index)
    pv = p[mask]
    n = len(pv)
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    placed = np.empty(n)
    placed[order] = adj
    out[np.flatnonzero(mask)] = placed
    return pd.Series(out, index=pvalues.index)


def _hist_js(left: np.ndarray, right: np.ndarray) -> float:
    both = np.concatenate([left, right])
    if both.size < 2 or np.nanmax(both) == np.nanmin(both):
        return np.nan
    bins = np.linspace(np.nanmin(both), np.nanmax(both), 21)
    h_left, _ = np.histogram(left, bins=bins, density=False)
    h_right, _ = np.histogram(right, bins=bins, density=False)
    h_left = h_left.astype(float) + 1e-12
    h_right = h_right.astype(float) + 1e-12
    return float(jensenshannon(h_left, h_right, base=2))


def compare_distributions(profile: pd.DataFrame) -> pd.DataFrame:
    """Каждая группа против остальных клиентов того же разреза."""
    rows = []
    proxy_col = profile.get("business_size_is_proxy")
    size_is_proxy = proxy_col is not None and pd.to_numeric(proxy_col, errors="coerce").fillna(0).eq(1).any()
    for slice_name, col in SLICES:
        if col not in profile.columns:
            continue
        work = profile.copy()
        work["_slice"] = work[col].fillna("не заполнен").astype(str)
        counts = work["_slice"].value_counts()
        usable = [v for v, n in counts.items() if n >= MIN_CLIENTS and (len(work) - n) >= MIN_CLIENTS]
        metrics = [m for m in METRICS if m in work.columns]
        if size_is_proxy and "размер" in slice_name:
            metrics = [m for m in metrics if m not in SIZE_BUILT_FROM]
        for value in usable:
            left_mask = work["_slice"].eq(value)
            for metric in metrics:
                left = pd.to_numeric(work.loc[left_mask, metric], errors="coerce").dropna().to_numpy()
                right = pd.to_numeric(work.loc[~left_mask, metric], errors="coerce").dropna().to_numpy()
                if len(left) < MIN_CLIENTS or len(right) < MIN_CLIENTS:
                    continue
                if np.unique(left).size < 2 and np.unique(right).size < 2:
                    continue
                ks = stats.ks_2samp(left, right)
                mw = stats.mannwhitneyu(left, right, alternative="two-sided")
                rows.append(
                    {
                        "slice_name": slice_name,
                        "slice_value": value,
                        "metric": metric,
                        "n_group": int(len(left)),
                        "n_rest": int(len(right)),
                        "enough": enough_label(int(left_mask.sum())),
                        "ks_stat": float(ks.statistic),
                        "ks_pvalue": float(ks.pvalue),
                        "mannwhitney_pvalue": float(mw.pvalue),
                        "wasserstein": float(stats.wasserstein_distance(left, right)),
                        "jensen_shannon": _hist_js(left, right),
                    }
                )
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    pieces = []
    for _, part in out.groupby(["slice_name", "metric"], sort=False):
        part = part.copy()
        part["ks_pvalue_bh"] = bh_adjust(part["ks_pvalue"])
        part["mannwhitney_pvalue_bh"] = bh_adjust(part["mannwhitney_pvalue"])
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True)


def association_tests(profile: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, left_col, right_col in ASSOCIATIONS:
        if left_col not in profile.columns or right_col not in profile.columns:
            continue
        frame = profile[[left_col, right_col]].copy()
        frame[left_col] = frame[left_col].fillna("не заполнен").astype(str)
        frame[right_col] = frame[right_col].fillna("не заполнен").astype(str)
        frame = frame.loc[~frame[left_col].isin(["не заполнен"])]
        frame = frame.loc[~frame[right_col].isin(["не заполнен", "нет оборота"])]
        keep_left = frame[left_col].value_counts()
        keep_right = frame[right_col].value_counts()
        frame = frame.loc[frame[left_col].isin(keep_left[keep_left >= 5].index)]
        frame = frame.loc[frame[right_col].isin(keep_right[keep_right >= 5].index)]
        if frame[left_col].nunique() < 2 or frame[right_col].nunique() < 2:
            continue
        table = pd.crosstab(frame[left_col], frame[right_col])
        chi2, pvalue, dof, _ = stats.chi2_contingency(table)
        r, c = table.shape
        n = int(table.to_numpy().sum())
        cramer = np.sqrt(chi2 / (n * (min(r, c) - 1))) if n and min(r, c) > 1 else np.nan
        rows.append(
            {
                "association": name,
                "n_clients": n,
                "n_rows": int(r),
                "n_cols": int(c),
                "chi2": float(chi2),
                "dof": int(dof),
                "pvalue": float(pvalue),
                "cramers_v": float(cramer),
            }
        )
    return pd.DataFrame(rows)


def cosine_by_slice(profile: pd.DataFrame) -> pd.DataFrame:
    """Средний профиль группы после z-нормализации, затем косинус между группами."""
    rows = []
    feature_cols = [m for m in METRICS if m in profile.columns]
    if len(feature_cols) < 2:
        return pd.DataFrame()
    usable = profile.copy()
    for col in feature_cols:
        usable[col] = pd.to_numeric(usable[col], errors="coerce")
    usable[feature_cols] = usable[feature_cols].fillna(usable[feature_cols].median())
    vary = [c for c in feature_cols if usable[c].std(ddof=0) > 0]
    if len(vary) < 2:
        return pd.DataFrame()
    z = usable[vary]
    scaled = (z - z.mean()) / z.std(ddof=0)
    scaled = pd.concat([usable[["inn"]].reset_index(drop=True), scaled.reset_index(drop=True)], axis=1)
    base = usable.reset_index(drop=True)
    for slice_name, col in SLICES:
        if col not in base.columns:
            continue
        labels = base[col].fillna("не заполнен").astype(str)
        counts = labels.value_counts()
        keep = counts[counts >= MIN_CLIENTS].index
        if len(keep) < 2:
            continue
        block = scaled.copy()
        block["_slice"] = labels.to_numpy()
        means = block.loc[block["_slice"].isin(keep)].groupby("_slice")[vary].mean()
        norms = np.linalg.norm(means.to_numpy(), axis=1)
        good = norms > 0
        means = means.loc[good]
        norms = norms[good]
        if len(means) < 2:
            continue
        unit = means.to_numpy() / norms[:, None]
        sim = unit @ unit.T
        names = list(means.index)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                rows.append(
                    {
                        "slice_name": slice_name,
                        "group_a": names[i],
                        "group_b": names[j],
                        "n_a": int(counts[names[i]]),
                        "n_b": int(counts[names[j]]),
                        "cosine_similarity": float(sim[i, j]),
                    }
                )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["slice_name", "cosine_similarity"])

with imp:
    profile = imp.fetch("select * from sandbox_ai.tmp_shestopalov_te_profile")

before = len(profile)
profile = profile.loc[profile["mcc_code"].astype(str) != "не заполнен"].copy()
print(f"Клиентов в профиле: {before}. Без кода точки убрано: {before - len(profile)}. В расчёте: {len(profile)}.")
profile.head()


## Численность групп

`достаточно` — от 100 клиентов, `на грани` — от 30 до 99, `мало` — меньше 30. Тесты распределений и косинус считаются только там, где клиентов не меньше 30, и в дополнении группы тоже не меньше 30.


In [ ]:
coverage = coverage_table(profile)
summary = (
    coverage.groupby(["slice_name", "enough"], as_index=False)
    .agg(groups=("slice_value", "count"), clients=("client_cnt", "sum"))
    .sort_values(["slice_name", "enough"])
)
display(summary)
usable = coverage.loc[coverage["enough"] != "мало"].copy()
print(f"Групп, где клиентов не меньше 30: {len(usable)}.")
display(usable)


## 1. Сравнение распределений

Каждая группа сравнивается с остальными клиентами того же разреза. Колмогоров–Смирнов и Манн–Уитни показывают, различаются ли распределения. Wasserstein — насколько далеко сдвинуты суммы и количества. Jensen–Shannon — насколько различается форма. p-значения Колмогорова–Смирнова и Манна–Уитни поправлены по Бенджамини–Хохбергу внутри разреза и метрики.

Ниже сначала строки, где после поправки p < 0.05, с самым большим сдвигом Wasserstein. Затем хи-квадрат и V Крамера: связаны ли группа MCC и регион, группа MCC и размер бизнеса, класс ОКВЭД и регион, класс ОКВЭД и размер. Это связь категорий, не сравнение сумм чека.


In [ ]:
tests = compare_distributions(profile)
if tests.empty:
    print("Нет группы, где и она сама, и остальные имеют не меньше 30 клиентов.")
else:
    strong = tests.loc[tests["ks_pvalue_bh"] < 0.05].sort_values("wasserstein", ascending=False)
    print(f"Сравнений: {len(tests)}. После поправки p < 0.05: {len(strong)}.")
    cols = ["slice_name", "slice_value", "metric", "n_group", "n_rest", "wasserstein", "jensen_shannon", "ks_pvalue_bh", "mannwhitney_pvalue_bh"]
    display(strong[cols].head(30))

assoc = association_tests(profile)
if assoc.empty:
    print("Хи-квадрат не посчитан: нет двух категорий с достаточным числом клиентов.")
else:
    display(assoc)


## 2. Нормализация и косинус

Поля портрета приводятся к одному масштабу (z-нормализация по всем клиентам расчёта). Для каждой группы берётся средний портрет. Косинус близок к 1, когда средние портреты похожи, и к 0, когда они расходятся. Сравниваются группы, а не отдельные клиенты. В пару попадают только группы от 30 клиентов.


In [ ]:
cosine = cosine_by_slice(profile)
if cosine.empty:
    print("Косинус не посчитан: в разрезе нет двух групп от 30 клиентов.")
else:
    pieces = []
    for slice_name, part in cosine.groupby("slice_name"):
        ordered = part.sort_values("cosine_similarity")
        far = ordered.head(5).assign(pair="далёкие")
        near = ordered.tail(5).assign(pair="близкие")
        pieces.append(pd.concat([far, near], ignore_index=True))
    show = pd.concat(pieces, ignore_index=True)
    display(show[["slice_name", "pair", "group_a", "group_b", "n_a", "n_b", "cosine_similarity"]])
